In [1]:
import pandas as pd
import csv
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

In [2]:
# Read in FL data and re-format date to the correct format
FL = pd.read_csv('../Data/florida.csv', delimiter=",")
FL.rename(columns={'date': 'time'}, inplace=True)
FL['time'] = pd.to_datetime(FL['time'])
FL['date'] = FL['time'].dt.date
display(FL.head(3))

# Agg VADAR score based on business id and dates
agg_FL=FL.groupby(['business_id', 'date'])['doc_sentiment'].agg(avg_VADAR='mean').reset_index()
agg_FL = agg_FL.merge(FL, on=['business_id', 'date'])[['business_id','business_name','city','state','date','avg_VADAR']]
display(agg_FL.head(3))

print('Total Biz IDs:',len(agg_FL['business_id'].unique()))

,business_id,business_name,city,state,latitude,longitude,stars,total_review_count,review_id,rating,time,review,doc_sentiment,aspect_sentiments,date
0,i-n7LKMjQsH94wfW8FNMBg,Spongeorama,Tarpon Springs,FL,28.155859,-82.758763,3.5,87,RUAWUgZHGdd88LMqgcXC1A,5.0,2005-03-18 01:49:32,Tarpon Springs feels almost like a foreign cou...,0.9840,"{'PRODUCT': {}, 'PERSON': {'Grime': -0.36}, 'O...",2005-03-18
1,8RwPYPVmudJP_LrdPdJMyw,Golden Corral Buffet & Grill,Largo,FL,27.893926,-82.778153,2.0,71,M5wbCApV0_xzLVb9yMJyZg,2.0,2005-07-14 18:31:07,One should be wary of all-you-can-eat experien...,0.9810,"{'PRODUCT': {}, 'PERSON': {}, 'ORG': {}}",2005-07-14
2,oUH2RAzsGa98XaeHqbxsBg,Sweet Tomatoes,Largo,FL,27.892672,-82.786626,4.0,104,0spvvdV_Wt1M8FpDZfijPA,4.0,2005-07-14 18:40:04,"Soup and salads are the main draw here, and th...",0.9659,"{'PRODUCT': {}, 'PERSON': {'Foccacio': 0.0}, '...",2005-07-14


,business_id,business_name,city,state,date,avg_VADAR
0,---kPU91CF4Lq2-WlRu9Lw,Frankie's Raw Bar,New Port Richey,FL,2020-01-29,0.9594
1,---kPU91CF4Lq2-WlRu9Lw,Frankie's Raw Bar,New Port Richey,FL,2020-02-11,0.9059
2,---kPU91CF4Lq2-WlRu9Lw,Frankie's Raw Bar,New Port Richey,FL,2020-02-20,0.9568


Total Biz IDs: 26330


In [3]:
######## Run on all businesses #######

# List of unique business IDs.
#n=3
all_business_ids = agg_FL['business_id'].unique()
random_n = agg_FL[agg_FL['business_id'].isin(all_business_ids)].copy()
random_n['date_index'] = pd.to_datetime(random_n['date'])
random_n.set_index('date_index', inplace=True)

random_n_vadar=random_n.copy()[['business_id','avg_VADAR','business_name']]
business_ids = random_n_vadar['business_id'].unique()
sample_freq='6M'

# Initialize a variable to keep track of overall accuracy.
overall_accuracy = []

test_predictions_df = pd.DataFrame()
predictions_df=pd.DataFrame()

# Group data by index
for i,business_id in enumerate(business_ids):

    business_data = random_n_vadar[random_n_vadar['business_id'] == business_id]
    
    # Group data by the half_year period and calculate the mean to reduce noise
    HalfYear_grouped = business_data.resample(sample_freq).mean(numeric_only=True)

    #Fill NaN values with the average of the previous and following values if there are still NaN
    #HalfYear_grouped_filled = HalfYear_grouped.fillna(method='ffill').fillna(method='bfill')
    HalfYear_grouped_filled = HalfYear_grouped.ffill().bfill()
    
    # Skip the business_id if there are NaNs or too few records to run ARIMA
    if HalfYear_grouped_filled['avg_VADAR'].isna().all() or len(HalfYear_grouped_filled)<5:
        continue

    # Calculate the index for the 80% train-test split.
    split_index = int(0.8 * len(HalfYear_grouped_filled ))
    
    # Split the data into training and testing sets.
    train_data = HalfYear_grouped_filled.iloc[:split_index]
    test_data = HalfYear_grouped_filled.iloc[split_index:]

    # Use auto_arima to optimize both trend and order
    trend_values = ['n', 'c', 't', 'ct']
    best_mse=float('inf')
    for trend_value in trend_values:
        best_model = auto_arima(
            train_data,
            seasonal=True,
            stepwise=True,
            suppress_warnings=True,
            trend=trend_value,
            error_action='ignore')

        # Make predictions on the test data using the best model
        test_predictions = best_model.predict(n_periods=len(test_data))

        # Calculate the MSE for hyperparameters selection
        mse_current = mean_squared_error(test_data['avg_VADAR'], test_predictions)

        if mse_current< best_mse:
            best_mse = mse_current
            best_trend = trend_value
            best_order=best_model.get_params()['order']

    print('best trend',best_trend)
    print('best order',best_order)
    print('-------------------------')

    test_predictions_data = pd.DataFrame({
        'Date': test_data.index,
        'Business_ID': business_id,
        'Actual_VADAR': test_data['avg_VADAR'],
        'Predicted_VADAR': test_predictions
    })
    test_predictions_df = pd.concat([test_predictions_df, test_predictions_data], ignore_index=True)

    # Append the accuracy to the overall_accuracy list.
    overall_accuracy.append(best_mse)

    # Make prediction based on all data
    best_model = auto_arima(
            HalfYear_grouped_filled,
            seasonal=True,
            stepwise=True,
            suppress_warnings=True,
            trend=best_trend,
            error_action='ignore')
    predictions = best_model.predict(n_periods=1)
    
    predictions_data = pd.DataFrame({
        'Business_ID': business_id,
        'Date': HalfYear_grouped_filled.index[-1]+ pd.DateOffset(months=6),
        'Predicted_VADAR': predictions
    })
    predictions_df = pd.concat([predictions_df, predictions_data], ignore_index=True)

display(test_predictions_df.head(5))
display(predictions_df.head(5))


best trend t
best order (1, 0, 1)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend t
best order (0, 0, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend ct
best order (1, 0, 1)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend n
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-

c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 1, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 1)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend t
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend n
best order (0, 0, 0)
---

c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


best trend n
best order (0, 0, 0)
-------------------------
best trend n
best order (0, 0, 1)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend t
best order (1, 0, 1)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend t
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-----

...

c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


best trend ct
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend ct
best order (1, 0, 1)
-------------------------
best trend ct
best order (3, 1, 1)
-------------------------
best trend ct
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend t
best order (1, 0, 1)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend ct
best order (0, 0, 1)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend t
best order (2, 0, 1)
-------------------------
best trend c
best order (0, 0, 1)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend t
best order (0, 1, 1)

c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '
c:\Users\Julie\AppData\Local\Programs\Python\Python312\Lib\site-packages\pmdarima\arima\auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


best trend ct
best order (0, 0, 0)
-------------------------
best trend ct
best order (1, 1, 1)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 1)
-------------------------
best trend t
best order (0, 0, 0)
-------------------------
best trend ct
best order (0, 0, 0)
-------------------------
best trend n
best order (1, 0, 0)
-------------------------
best trend c
best order (1, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
-------------------------
best trend c
best order (0, 0, 0)
-------------------------
best trend n
best order (0, 0, 0)
-------------------------
best trend c
best order (2, 0, 0)
-------------------------
best trend t
best order (1, 0, 0)
--

ValueError: Input contains NaN.

In [ ]:
# Calculate the average accuracy across all businesses.
average_accuracy = np.mean(overall_accuracy)
print(f"Average Mean Squared Error (MSE) for businesses: {average_accuracy}")

In [5]:
print('Total Num of Test Predictions:',len(test_predictions_df))
print('Test Prediction, Num of Vadar >1:',len(test_predictions_df[(test_predictions_df['Predicted_VADAR']<-1)]))
print('Test Prediction, Num of Vadar <-1:',len(test_predictions_df[(test_predictions_df['Predicted_VADAR']>1)]))
print('--------------------')
print(' ')

print('Num of biz IDs used in prediction: ',len(predictions_df['Business_ID'].unique()))
print('Num of skipped biz IDs due to insufficient info: ',len(all_business_ids)-len(predictions_df['Business_ID'].unique()))
print('Overall Prediction, Num of Vadar >1:',len(predictions_df[(predictions_df['Predicted_VADAR']<-1)]))
print('Overall Prediction, Num of Vadar <-1:',len(predictions_df[(predictions_df['Predicted_VADAR']>1)]))


Total Num of Test Predictions: 13
Test Prediction, Num of Vadar >1: 0
Test Prediction, Num of Vadar <-1: 3
--------------------
 
Num of biz IDs used in prediction:  3
Num of skipped biz IDs due to insufficient info:  0
Overall Prediction, Num of Vadar >1: 0
Overall Prediction, Num of Vadar <-1: 0


In [7]:
test_predictions_df.to_csv('../Data/FL_test_predictions.csv')
predictions_df.to_csv('../Data/FL_predictions.csv')